# Activity 3 — Colab–VS Code Bridge
**Curso:** TI2006B · **Alumno:** Jorge López Martínez (A01385731)

## Pregunta
En un negocio pequeño que agenda citas, ¿qué día de la semana tiene la **tasa de inasistencia (no-show)** más alta, y la anticipación con la que se reserva se relaciona con faltar?

## Procedencia de los datos
Los datos son **100 % sintéticos**: se generan en este mismo notebook con `numpy` y una semilla fija (`SEED = 42`). No se usan archivos locales, Google Drive ni datos personales, confidenciales o reales.

## Resultado esperado
1. Una tabla con la tasa de no-show por día de la semana.
2. Una comparación de la tasa de no-show según la anticipación de la reserva.
3. Una gráfica de barras y una conclusión breve.

## Cómo reproducir
Abre el notebook desde GitHub en Colab → *Runtime → Restart session and run all*. No requiere instalar nada: solo usa `numpy`, `pandas` y `matplotlib`, que ya vienen en Colab.

## 1. Setup y dependencias

In [ ]:
import sys, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Si algún paquete faltara en otro entorno, descomenta:
# %pip install numpy pandas matplotlib

print("Python     :", sys.version.split()[0])
print("Plataforma :", platform.platform())
print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
try:
    import google.colab  # noqa: F401
    print("Runtime    : Google Colab")
except ImportError:
    print("Runtime    : local (no Colab)")

## 2. Datos sintéticos
Se generan 1,000 citas ficticias. La probabilidad de faltar aumenta con la anticipación de la reserva y es un poco mayor los lunes y viernes (supuesto inventado para el ejercicio).

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
N = 1_000

dias = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb"]
dia = rng.choice(dias, size=N)
anticipacion = rng.integers(0, 31, size=N)          # días entre reserva y cita

efecto_dia = pd.Series({"Lun": 0.05, "Mar": 0.0, "Mié": 0.0,
                        "Jue": 0.0, "Vie": 0.06, "Sáb": 0.02})
p_falta = 0.08 + efecto_dia[dia].to_numpy() + 0.004 * anticipacion
no_show = rng.random(N) < p_falta

citas = pd.DataFrame({"dia": pd.Categorical(dia, categories=dias, ordered=True),
                      "anticipacion_dias": anticipacion,
                      "no_show": no_show})
print(citas.shape)
citas.head()

## 3. Análisis

In [ ]:
por_dia = (citas.groupby("dia", observed=True)["no_show"]
                .agg(citas="size", tasa_no_show="mean")
                .round(3))
print("Tasa de no-show por día:")
display(por_dia)

citas["anticipacion"] = pd.cut(citas["anticipacion_dias"], bins=[-1, 3, 14, 30],
                               labels=["0–3 días", "4–14 días", "15–30 días"])
por_anticipacion = (citas.groupby("anticipacion", observed=True)["no_show"]
                         .agg(citas="size", tasa_no_show="mean").round(3))
print("Tasa de no-show por anticipación:")
display(por_anticipacion)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
por_dia["tasa_no_show"].plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("No-show por día"); axes[0].set_ylabel("tasa"); axes[0].set_xlabel("")
por_anticipacion["tasa_no_show"].plot.bar(ax=axes[1], color="#DD8452")
axes[1].set_title("No-show por anticipación"); axes[1].set_xlabel("")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 4. Conclusión

In [ ]:
peor_dia = por_dia["tasa_no_show"].idxmax()
print(f"Día con mayor no-show: {peor_dia} ({por_dia.loc[peor_dia, 'tasa_no_show']:.1%})")
print(f"Reservas 0–3 días   : {por_anticipacion.loc['0–3 días', 'tasa_no_show']:.1%}")
print(f"Reservas 15–30 días : {por_anticipacion.loc['15–30 días', 'tasa_no_show']:.1%}")

**Interpretación.** En estos datos sintéticos, las citas reservadas con mucha anticipación tienen más inasistencias que las reservadas con pocos días, y los lunes/viernes destacan frente a mitad de semana. Una acción posible sería enviar recordatorios extra a citas reservadas con más de dos semanas de anticipación.

**Limitación:** los datos son inventados y el patrón fue programado en la simulación, así que el resultado ilustra el flujo de trabajo, no una conclusión sobre un negocio real.

## 5. Contenido guardado vs. estado del runtime
- **Se guarda en el archivo `.ipynb` (GitHub):** el código, el texto Markdown y las salidas que estaban visibles al guardar.
- **NO se guarda ni se comparte:** las variables en memoria, los paquetes instalados, la versión exacta de Python del runtime y los archivos locales o de Drive. Por eso el notebook genera sus propios datos y muestra las versiones en la celda de setup.